# A8. Can the data separate a shared driver from co-production? (PM2.5 - ozone association, Bogotá)

Two hypotheses remain for the PM2.5-ozone association after the fire-file correction (Section 3.4):
  H1, a SHARED DRIVER: PM2.5 and ozone both rise in the same dry, sunny, smoky season / synoptic conditions, so their association is
      seasonal or meteorological, not chemical.
  H2, CO-PRODUCTION: part of PM2.5 is secondary aerosol formed in the same photo-oxidation that makes ozone, so the association survives
      once season, smoke and primary emissions are removed and shows a timing (aerosol formation lagging precursors / ozone) that primary tracers do not.
Tests that discriminate (each one has a prediction under H1 and under H2):
  A. Within-season association: partial r (PM2.5, O3 | NO2, NO, SW, fire) after removing the month means, within each season, and on low-fire days.
     H1 predicts the association collapses once season is removed; H2 predicts it survives.
  B. Hourly lead-lag of PM2.5 against ozone compared with primary tracers (PM10, NO2, NO, SO2) and with the coarse fraction (PM10 - PM2.5).
     If PM2.5 leads ozone by 3-4 h because traffic emissions peak in the morning, every primary tracer should show the same lead (H1 / timing artefact);
     a lead specific to PM2.5, surviving a control for NO2, NO and radiation, favours H2.
  C. Afternoon PM2.5 increment (13-17 h minus 09-11 h) against that day's ozone, net of the same increments of PM10 and NO2 (dilution and primary changes).
Moving-block bootstrap (30-day blocks, 1000 replicates) for the daily tests.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import glob, warnings
import numpy as np, pandas as pd, xarray as xr
from style import ROOT, DATA_OUT
warnings.filterwarnings("ignore")
P = ROOT / "Data/processed/colombia/bogota"
SAVE = True

In [2]:
# Settings (edit here)
N_BOOT, BLOCK = 1000, 30
DAYTIME = (6, 19)
LAGS = list(range(-6, 7))
SEASONS = {"Dec-Mar (dry, smoke)": [12, 1, 2, 3], "Apr-May": [4, 5], "Jun-Sep (low smoke)": [6, 7, 8, 9], "Oct-Nov": [10, 11]}

In [3]:
# A. Daily: association within season and on low-fire days
d = pd.read_parquet(P / "bogota_o3_master_dataset.parquet"); d["date"] = pd.to_datetime(d["date"])
d["fire_eq1"] = d["hotspots_sector_E_lag2"] + d["hotspots_sector_N"]
need = ["pm25", "o3", "no2", "no", "co", "temp_2m", "rh_2m", "blh_mean", "radiation_mean", "fire_eq1"]
d = d.dropna(subset=need).reset_index(drop=True); d["month"] = d.date.dt.month
def resid(y, X):
    A = np.column_stack([np.ones(len(y))] + ([X] if X.ndim == 1 else [X[:, i] for i in range(X.shape[1])])) if X is not None and X.size else np.ones((len(y), 1))
    return y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
def pcorr(df, ctrl):
    X = df[ctrl].to_numpy(float) if ctrl else None
    return float(np.corrcoef(resid(df.pm25.to_numpy(float), X), resid(df.o3.to_numpy(float), X))[0, 1])
def block_ci(df, ctrl, rng):
    n = len(df); nb = int(np.ceil(n / BLOCK)); out = []
    for _ in range(N_BOOT):
        st = rng.integers(0, n - BLOCK + 1, nb); idx = np.concatenate([np.arange(s, s + BLOCK) for s in st])[:n]
        out.append(pcorr(df.iloc[idx], ctrl))
    return np.percentile(out, [2.5, 97.5])
rng = np.random.default_rng(1)
CTRL = ["no2", "no", "radiation_mean", "fire_eq1"]
# month-demeaned copies: removes the seasonal cycle of every variable
dm = d.copy()
for c in set(CTRL + ["pm25", "o3"]): dm[c] = d[c] - d.groupby("month")[c].transform("mean")
rows = []
def add(label, df, ctrl, n_extra=""):
    r = pcorr(df, ctrl); lo, hi = block_ci(df, ctrl, rng); rows.append(dict(sample=label, controls="+".join(ctrl) or "none", n=len(df), partial_r=r, ci_lo=lo, ci_hi=hi))
add("all days", d, []); add("all days", d, ["no2", "no", "radiation_mean"]); add("all days", d, CTRL)
add("all days, month means removed", dm, []); add("all days, month means removed", dm, ["no2", "no", "radiation_mean"]); add("all days, month means removed", dm, CTRL)
for lab, ms in SEASONS.items():
    s = d[d.month.isin(ms)].reset_index(drop=True)
    add(f"{lab}", s, []); add(f"{lab}", s, ["no2", "no", "radiation_mean"]); add(f"{lab}", s, CTRL)
lowfire = d[d.fire_eq1 <= d.fire_eq1.quantile(0.25)].reset_index(drop=True)
add("lowest fire quartile", lowfire, []); add("lowest fire quartile", lowfire, ["no2", "no", "radiation_mean"])
lowfire_m = dm.loc[d.fire_eq1 <= d.fire_eq1.quantile(0.25)].reset_index(drop=True)
add("lowest fire quartile, month means removed", lowfire_m, ["no2", "no", "radiation_mean"])
resA = pd.DataFrame(rows); print(resA.round(3).to_string(index=False))
if SAVE: resA.to_csv(DATA_OUT / "coproduction_daily_within_season.csv", index=False)

                                   sample                       controls    n  partial_r  ci_lo  ci_hi
                                 all days                           none 2917      0.493  0.400  0.574
                                 all days          no2+no+radiation_mean 2917      0.512  0.416  0.591
                                 all days no2+no+radiation_mean+fire_eq1 2917      0.312  0.228  0.386
            all days, month means removed                           none 2917      0.323  0.240  0.395
            all days, month means removed          no2+no+radiation_mean 2917      0.387  0.303  0.461
            all days, month means removed no2+no+radiation_mean+fire_eq1 2917      0.282  0.204  0.353
                     Dec-Mar (dry, smoke)                           none  965      0.536  0.415  0.643
                     Dec-Mar (dry, smoke)          no2+no+radiation_mean  965      0.579  0.482  0.666
                     Dec-Mar (dry, smoke) no2+no+radiation_mean+fire_eq1 

In [4]:
# B. Hourly lead-lag of PM2.5 and of primary tracers against ozone
def rd(name, col):
    x = pd.read_excel(ROOT / f"Data/COLOMBIA/Hourmean/{name}_hourmean.xlsx", usecols=["Dates", "Bogota"]).rename(columns={"Dates": "t", "Bogota": col})
    x["t"] = pd.to_datetime(x["t"]); return x.drop_duplicates("t").set_index("t")
h = rd("PM25", "pm25")
for nm, col in (("O3", "o3"), ("PM10", "pm10"), ("NO2", "no2"), ("NO", "no"), ("SO2", "so2")): h = h.join(rd(nm, col), how="outer")
h = h.sort_index().asfreq("h"); h["pmc"] = h.pm10 - h.pm25
ds = [xr.open_dataset(p) for p in sorted(glob.glob(str(ROOT / "Data/external/era5/bogota_era5_ssrd_*.nc")))]
ds = [x.rename({"valid_time": "time"}) if "valid_time" in x.dims else x for x in ds]
rad = xr.concat(sorted(ds, key=lambda x: x.time.values[0]), dim="time").mean(dim=["latitude", "longitude"])["ssrd"] / 3600.0
rad = rad.to_series(); rad.index = rad.index + pd.Timedelta(hours=-5); rad = rad[~rad.index.duplicated()]
h["rad"] = rad.reindex(h.index); h["hour"] = h.index.hour; h["month"] = h.index.month
VARS = ["pm25", "pm10", "pmc", "no2", "no", "so2", "o3", "rad"]
for c in VARS: h[c + "_a"] = h[c] - h.groupby("hour")[c].transform("mean")
def xcorr(x, lag, mask=None, control=None):
    y = h["o3_a"].shift(-lag); df = pd.DataFrame({"x": h[x + "_a"], "y": y, "hh": h.hour, "m": h.month})
    for c in (control or []): df[c] = h[c + "_a"].shift(-lag) if c == "rad" else h[c + "_a"]
    df = df.dropna(); df = df[(df.hh >= DAYTIME[0]) & (df.hh <= DAYTIME[1])]
    if mask is not None: df = df[df.m.isin(mask)]
    if control:
        A = np.column_stack([np.ones(len(df))] + [df[c].to_numpy() for c in control])
        ex = df.x.to_numpy() - A @ np.linalg.lstsq(A, df.x.to_numpy(), rcond=None)[0]; ey = df.y.to_numpy() - A @ np.linalg.lstsq(A, df.y.to_numpy(), rcond=None)[0]
        return float(np.corrcoef(ex, ey)[0, 1]), len(df)
    return float(np.corrcoef(df.x, df.y)[0, 1]), len(df)
lagrows = []
for x in ["pm25", "pm10", "pmc", "no2", "no", "so2"]:
    for lag in LAGS:
        r, n = xcorr(x, lag); lagrows.append(dict(series=x, lag_h=lag, r=r, n=n, subset="all months", controls="none"))
for x in ["pm25", "pmc", "no2"]:
    for lag in LAGS:
        for lab, ms in (("Dec-Mar", SEASONS["Dec-Mar (dry, smoke)"]), ("Jun-Sep", SEASONS["Jun-Sep (low smoke)"])):
            r, n = xcorr(x, lag, mask=ms); lagrows.append(dict(series=x, lag_h=lag, r=r, n=n, subset=lab, controls="none"))
for lag in LAGS:   # PM2.5 with NO2, NO and radiation (same-hour anomalies) regressed out of both series
    r, n = xcorr("pm25", lag, control=["no2", "no", "rad"]); lagrows.append(dict(series="pm25", lag_h=lag, r=r, n=n, subset="all months", controls="no2+no+rad"))
    r, n = xcorr("pm25", lag, control=["pm10"]); lagrows.append(dict(series="pm25", lag_h=lag, r=r, n=n, subset="all months", controls="pm10"))
lag = pd.DataFrame(lagrows)
piv = lag[(lag.subset == "all months") & (lag.controls == "none")].pivot(index="lag_h", columns="series", values="r"); print(piv.round(3).to_string())
peak = lag.loc[lag.groupby(["series", "subset", "controls"]).r.idxmax()][["series", "subset", "controls", "lag_h", "r"]]; print(peak.round(3).to_string(index=False))
if SAVE: lag.to_csv(DATA_OUT / "coproduction_hourly_leadlag.csv", index=False); peak.to_csv(DATA_OUT / "coproduction_hourly_peaks.csv", index=False)

series     no    no2   pm10   pm25    pmc    so2
lag_h                                           
-6     -0.159  0.190 -0.084  0.038 -0.176 -0.224
-5     -0.162  0.205 -0.057  0.065 -0.155 -0.231
-4     -0.166  0.211 -0.029  0.089 -0.130 -0.235
-3     -0.170  0.209 -0.001  0.110 -0.104 -0.235
-2     -0.174  0.200  0.022  0.128 -0.081 -0.232
-1     -0.179  0.193  0.039  0.146 -0.069 -0.228
 0     -0.180  0.201  0.049  0.162 -0.068 -0.222
 1     -0.152  0.230  0.053  0.178 -0.075 -0.214
 2     -0.102  0.256  0.057  0.192 -0.081 -0.207
 3     -0.052  0.267  0.059  0.201 -0.085 -0.202
 4     -0.019  0.262  0.056  0.201 -0.090 -0.195
 5     -0.004  0.246  0.047  0.192 -0.097 -0.186
 6      0.003  0.225  0.034  0.176 -0.106 -0.177
series     subset   controls  lag_h      r
    no all months       none      6  0.003
   no2    Dec-Mar       none      4  0.305
   no2    Jun-Sep       none      3  0.309
   no2 all months       none      3  0.267
  pm10 all months       none      3  0.059
  pm25 

In [5]:
# C. Afternoon PM2.5 increment against the day's ozone
hh = h.copy(); hh["date"] = hh.index.date
def win(c, a, b): return hh[(hh.hour >= a) & (hh.hour <= b)].groupby("date")[c].mean()
inc = pd.DataFrame({f"d_{c}": win(c, 13, 17) - win(c, 9, 11) for c in ("pm25", "pm10", "no2", "no")})
inc["o3_max"] = hh.groupby("date")["o3"].max(); inc["o3_pm_hours"] = win("o3", 13, 17); inc["pm_morning"] = win("pm25", 6, 8); inc["rad"] = win("rad", 11, 15)
inc.index = pd.to_datetime(inc.index); inc = inc.dropna()
dd = inc.join(d.set_index("date")[["fire_eq1", "co", "wind_speed_era5"]], how="inner").dropna()
def pc2(df, x, y, ctrl):
    X = df[ctrl].to_numpy(float) if ctrl else None
    return float(np.corrcoef(resid(df[x].to_numpy(float), X), resid(df[y].to_numpy(float), X))[0, 1])
cres = []
for ctrl in ([], ["d_pm10"], ["d_pm10", "d_no2", "d_no"], ["d_pm10", "d_no2", "d_no", "pm_morning", "rad"], ["d_pm10", "d_no2", "d_no", "pm_morning", "rad", "fire_eq1", "wind_speed_era5"]):
    cres.append(dict(controls="+".join(ctrl) or "none", n=len(dd), partial_r_dPM25_vs_O3_afternoon=pc2(dd, "d_pm25", "o3_pm_hours", ctrl), partial_r_dPM25_vs_O3_max=pc2(dd, "d_pm25", "o3_max", ctrl)))
cres = pd.DataFrame(cres); print(cres.round(3).to_string(index=False))
if SAVE: cres.to_csv(DATA_OUT / "coproduction_afternoon_increment.csv", index=False)

                                                 controls    n  partial_r_dPM25_vs_O3_afternoon  partial_r_dPM25_vs_O3_max
                                                     none 2453                           -0.054                     -0.069
                                                   d_pm10 2453                           -0.118                     -0.112
                                        d_pm10+d_no2+d_no 2453                           -0.066                     -0.067
                         d_pm10+d_no2+d_no+pm_morning+rad 2453                            0.026                      0.034
d_pm10+d_no2+d_no+pm_morning+rad+fire_eq1+wind_speed_era5 2453                            0.020                      0.025
